# Python Crash Course - Lecture 11

Contents:
* __Recap of Pandas part 1__
* __Pandas part 2__
    * changing values of cells
    * grouping things
    * merging/joining things
    * more on applying functions
* __(At the end) code for binary search__

***
# Recap of Pandas part 1

In [ ]:
import pandas as pd # traditionally imported as pd, but you are free to use any acronym you like
import numpy as np

### pandas functions

```python
pd.read_csv() # read data from file

pd.DataFrame() # create dataframe

pd.concat() # concatenate dataframes

pd.to_csv() # write data to file
```

### pandas DataFrame attributes

``` python
.index # get row indexes

.column # get column names

.shape # get dataframe shape (rows, columns)
```

### pandas methods for DataFrames

```python
.head() # get top rows (default 5)

.tail() # get last rows (default 5) 

.describe(). # calcualte summary statistics

.sort_values(by =...) # sort dataframe according to one or more columns

.drop() # drop columns or rows

.reset_index() # reset row indexes

.rename() # rename columns

.copy() # create copy of dataframe (can also be done on dataframe view)
```

### accessing content in DataFrames

```python
df[col_name] # access one column

df[[col1, col2, col3]] # access multiple columns

df.loc[row_id] # access one row

df.loc[[row_id1,row_id2,row_id3]] # access multiple rows

df.loc[[rowlabels], [columnlabels]] # access rows and columns
```

### Boolean indexing (filtering by condition)
__Conditions can be combined with "bitwise and" `&`, "bitwise or" `|` operators__

```python
df[columnlabel] # can be combined with comparison > < == != operators

df[condition] # returns only those rows where condition is True

df[(condition1) & (condition2)] # returns only those rows where both conditions are True

df[(condition1) | (condition2)] # returns only those rows at least one of the conditions is True
```

### Filtering missing values
```python
df[column].isna() # returns only those rows where row value is NaN

df[column].notna() # returns only those rows where row value is not NaN

~ # tilde operators means not and can be used with isna and notna
df[column].isna() == ~df[column].notna()

```

### Applying functions on DataFrames
```python
.apply(some_function) # Apply a function along an axis of the DataFrame
```


***
# Pandas Part 2
* changing values of cells
* grouping things
* merging/joining things
* more on applying functions

In [ ]:
# lets load the titanic data again
df = pd.read_csv("data/titanic.csv")

## Changing values of an individual cell

#### `df.loc[rowlabel, columnlabel] = new_value`

In [ ]:
df.head(1)

In [ ]:
# this changes the value in the cell with row_index = 0, column = "Name"
df.loc[0, "Name"] = "DiCaprio, Mr. Leonardo" # assigning CHANGES the object!
df.head(1)

In [ ]:
# this changes the values of the cells of the first 3 rows, column = "Name"
df.loc[[0,1,2], "Name"] = "There was room for Leo on the door!" # assigning CHANGES the object!
df.head(3)

## Changing values by replacing specific values with others

#### `df.replace(replace_stuff, replace_with)`, Note! It has a regex flag, which lets you use regex to catch strings

In [ ]:
df['Embarked'].replace('S','Southampton')

### replace returns a new DataFrame/Series - does not overwrite the original

In [ ]:
# you either overwrite the column
df['Embarked'] = df['Embarked'].replace('S','Southampton')

In [ ]:
# or use the inplace = True flag
df['Embarked'].replace('C','Cherbourg', inplace = True) # this approach will be deprecated 

## Replacing more than one value
#### `df.replace({column_name:{'replace_i':replace_with_i, replace_j: replace_with_j, ...}})`

In [ ]:
df = pd.read_csv("data/titanic.csv")
# safer alternative 
df.replace({'Embarked':{'S':'Southampton','C':'Cherbourg','Q':'Queenstown'}},inplace=True)

## Replacing missing values 
#### `df.fillna()` - replaces NaNs with specific values

In [ ]:
df.fillna('No info dude') # again this creates a view, use inplace = True if you want to save it, or overwrite the variable

#### Replacing NaNs with different values depending on the column
#### `df.fillna({column_1: value_1, column_2: value_2, ...})`

In [ ]:
df.fillna({"Age": np.mean(df['Age']), 
           "Cabin": 'The floor', 
           "Name": 'Unknown'})

__Very smilar functionality as `df.replace()`__

# Try it out yourself!
### SettingWithCopyWarning
The below line raises an error, fix it
```python
df[df["Age"]<18]["lifeboat"] = True
``` 


In [ ]:
# let's add a column, "lifeboat"...
df["lifeboat"] = None
# and let's assume all kids were taken to a lifeboat:
df[df["Age"]<18] # for these rows, we want lifeboat=True
df[df["Age"]<18]["lifeboat"] # so we need to set all these values to True
df[df["Age"]<18]["lifeboat"] = True
df.head(20) # why didn't it work?

#### solution
```python
df.loc[df["Age"]<18,"lifeboat"] = True
```

## Replace values according using conditions
### `df.where(condition, other = new_value)` - replace values where the condition is False.

In [ ]:
# relabel people below 20 as 'Young'
df['Age'].where(df['Age'] > 20, 'Young') # returns a series, remember to assign it to column or use inplace

### Be careful with using it on whole dataframes
#### It will replace all values for those rows which the condition does not capture to the new value

In [ ]:
# relabel people below 20 as 'Young'
df.where(df['Age'] > 20, 'Young') # returns a DataFrame

# Grouping things 😱
### Using the `df.groupby(by = column)` function
In Pandas its easy to group similar objects/rows and calcualte various things

<img src="files/group_by.png" alt="csv file" width=900px>

### Lets calculate the average price Titanic passengers paid for their tickets depending on `Pclass`

In [ ]:
df[['Pclass','Fare']].groupby(by = 'Pclass')
# the group by object creates a GroupBy object

### Curently Pandas has only grouped objects together, but you have to tell it what to do with values.

#### For example, we can calculate the average fare passangers paid in each class using `.mean()`

In [ ]:
df[['Pclass','Fare']].groupby(by = 'Pclass').mean() 
# average price paid by passengers in each class

#### Other functions are also available for Grouped objects, and can calculate: `.min(), .max(), .sum(), .count(), ...`)

In [ ]:
df[['Pclass','Fare']].groupby(by = 'Pclass').sum() 

### To calculate multiple things on the same grouped object

In [ ]:
df[['Pclass','Fare']].groupby(by = 'Pclass').agg({'Fare':['sum','max','mean'], 'Pclass':'count'})

### You can even define and use your own functions for the grouping! See example [here](https://discovery.cs.illinois.edu/guides/Modifying-DataFrames/custom_aggregation_functions/)

## Grouping over multiple columns
#### `df.groupby(by = [list_of_columns]).some_grouping_function()`

In [ ]:
df.groupby(['Pclass','Sex']).agg({'Pclass':'count'})

# Try it out yourself!

Use `.groupby()` to compute the mean `Age` and mean `Fare` of Survived (`Survived = 1`) vs. not-Survived (`Survived = 0`) passengers.

There are many ways to solve this, but you could do it with one single line of code!

In [ ]:
# YOUR CODE HERE

#### Solution
```python
df.groupby('Survived').agg({'Age':'mean','Fare':'mean'})
```

## Merging multiple DataFrames

#### We start with creating a new dataframe (`.DataFrame()`)
Let's say we want to create a dataset of favourite colors for different passenger classes

In [ ]:
a = [1,2,3] # passenger classes
colors = ['Gold','Blue','Orange'] # favourite colors
mascots = ['Donkey','Deer','Kangaroo']

In [ ]:
# putting the list into a DataFrame
df_more_info = pd.DataFrame({'Pclass':a, 'Class_color':colors, 'Class_mascot':mascots})
df_more_info

## Merging / joining dataframes
### `df1.merge(df2, how = ..., on = ...)` lets you join information from multiple dataframes together

In [ ]:
# lets see what we are working wth
df.head(0) # returns just the columns

In [ ]:
df_colors.head(0)

In [ ]:
df_merged = df.merge(df_more_info, how = 'inner', on = 'Pclass') # this returns a new DataFrame
df_merged.head(3)

### The `how` to merge input in  `.merge(how = 'option')`  is really important!
Different options are available, i.e. `how = 'left'`, `how = 'right'`, `how = 'outer'`, `how = 'inner'`

<img src="files/joins.png" alt="csv file" width=900px>

# <font color='red'> SHOW EXAMPLE ON SLIDES </font>

## [from the book: Thinking in Pandas] The `.apply()` method is one of the most incorrectly used functions in pandas. Chances are if you are using it, you shouldn’t be. 

__This is because apply "applies" the function to each row or each column in the data set effectively breaking one of the cardinal rules of using pandas: do not iterate over the data set.__

## Advanced data processing on columns
__There exist different functions for applying functions__ :
* `.map()` - is defined on Series only
* `.applymap()` - is defined on DataFrames only
* `.apply()` -  is defined on both

In [ ]:
# lets define a custom function 
def my_function(fare):
    if fare <  10:
        return 'poor'
    elif 10 <= fare < 50:
        return 'maybe poor'
    else: 
        return 'not poor'

In [ ]:
df['is_poor'] = df['Fare'].map(my_function) # this works for basic functions

__you could have achieved similar results using `.apply()`__

(but personally, my 'brain' prefers to use `.map()`)

In [ ]:
# its the same as this 
df['Fare'].apply(lambda x: my_function(x))

### I like `.map()` because it can take dictinaries as input
#### `df.map({'old_value1': 'new_value', 'old_value2': 'new_value', ...})`

In [ ]:
df['Survived'] = df['Survived'].map({0:'Dead',1:'Alive'})
df.head(3)

__Basically same functionality as `.replace()`__ 

# One final thing - Iterating over rows (not recommended ☠️), but can be done
### `.iterrows()` Its slower than Panda's native functions but can be nice to use for testing things

In [ ]:
for i, row in df.iterrows():
    print(i,row)
    break

***
# binary search

In [ ]:
def binary_search(array, target_number):
    L_index = 0
    R_index = len(array)-1

    # sort array
    array = sorted(array)
    
    while L_index <= R_index:
        guess_index = L_index + (R_index - L_index) // 2

        print('L: {}, R: {}, Guess = {}'.format(array[L_index], array[R_index], array[guess_index]))
        
        if array[guess_index] < target_number:
            L_index = guess_index + 1
            print('Target number is higher\n')
            
        elif array[guess_index] > target_number:
            R_index = guess_index - 1
            print('Target number is lower\n')
            
        else:
            print('Your number is {}'.format(array[guess_index]))
            break
            

In [ ]:
x = np.random.randint(0,1000000000,1000)
guess = np.random.choice(x)

In [ ]:
binary_search(np.arange(1,1000+1),311)